<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Chapter 2: Generating Text with a Pre-trained LLM 使用预训练的LLM生成文本

Packages that are being used in this notebook:

在本笔记本中使用的包：

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.2
torch version: 2.7.1
tokenizers version: 0.21.4


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F01_raschka.webp?1" width="500px">

&nbsp;
## 2.1 Introduction to LLMs for text generation 介绍用于文本生成的LLMs

- No code in this section
- How do LLMs generate text?
- This chapter is a setup chapter: setting up the coding environment and LLM we will be using throughout the book
- We also code text generation functions that we will use and extend in upcoming chapters
- 本节没有代码
- LLMs如何生成文本？
- 本章是设置章节：设置我们将在本书中使用的编码环境和LLM
- 我们还编写了将在后续章节中扩展的文本生成函数

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F02_raschka.webp?1" width="300px">

- LLM (and neural network) flowcharts are traditionally read and drawn from top to bottom
- LLM(和神经网络)流程图通常从上到下阅读和绘制

&nbsp;
## 2.2 Setting up the coding environment 设置编码环境

- If you are reading this book, you likely coded in Python before
- The simplest way to install dependencies, if you already have a Python environment set up (with Python 3.10 or newer), is to use `pip`:
- 如果您正在阅读本书，您可能之前已经使用Python编码过
- 如果您已经设置了一个Python环境（使用Python 3.10或更高版本），最简单的安装依赖项的方法是使用`pip`:

In [2]:
#!pip install -r https://raw.githubusercontent.com/rasbt/reasoning-from-scratch/refs/heads/main/requirements.txt

- For this chapter, dependencies can also be installed manually:
- 对于本章，依赖项也可以手动安装：

In [3]:
#!pip install torch>=2.7.1 tokenizers>=0.21.2

- My preferred way is to use the widely recommended [uv](https://docs.astral.sh/uv/) Python package and project manager
- To install `uv`, run the installation for your OS from the official website: https://docs.astral.sh/uv/getting-started/installation/
- Next, clone the GitHub repo:
- 我的首选方法是使用广泛推荐的[uv](https://docs.astral.sh/uv/)Python包和项目管理器
- 要安装`uv`，请从官方网站运行您的OS的安装程序：https://docs.astral.sh/uv/getting-started/installation/
- 接下来，克隆GitHub仓库：

In [4]:
#!git clone --depth 1 https://github.com/rasbt/reasoning-from-scratch.git

- If you don't have `git` installed, you can also manually download the source code repository from the Manning website or by clicking this link: https://github.com/rasbt/reasoning-from-scratch/archive/refs/heads/main.zip (unzip it after downloading)
- 如果您没有安装`git`，您也可以从Manning网站或通过点击此链接手动下载源代码仓库：https://github.com/rasbt/reasoning-from-scratch/archive/refs/heads/main.zip（下载后解压）

- In the terminal, navigate to the `reasoning-from-scratch` folder
- Run `uv run jupyter lab` to launch JupyterLab and open a blank notebook or the notebook for this chapter
- This command also sets up a local virtual environment (usually in `.venv/`) and installs all dependencies from the `pyproject.toml` file inside the `reasoning-from-scratch` folder automatically
- 在终端中，导航到`reasoning-from-scratch`文件夹
- 运行`uv run jupyter lab`启动JupyterLab并打开一个空白笔记本或本章笔记本
- 此命令还会自动设置本地虚拟环境（通常在`.venv/`中）并从`reasoning-from-scratch`文件夹中的`pyproject.toml`文件安装所有依赖项

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F03_raschka.webp?1" width="500px">

- See [../02_setup-tips/python-instructions.md](../02_setup-tips/python-instructions.md) for additional installation details and options if needed
- 如果需要，请参阅[../02_setup-tips/python-instructions.md](../02_setup-tips/python-instructions.md)以获取其他安装详细信息和选项

&nbsp;
## 2.3 Understanding hardware needs and recommendations 了解硬件需求和推荐

- If you are new to PyTorch, I recommend reading through my [PyTorch in One Hour: From Tensors to Training Neural Networks on Multiple GPUs](https://sebastianraschka.com/teaching/pytorch-1h/) tutorial
- If you followed the previous section, you should have PyTorch installed
- Check manually if your PyTorch installation supports GPU; see what's supported on your machine: 
- 如果您是PyTorch的新手，我建议您阅读我的[PyTorch in One Hour：从张量到在多个GPU上训练神经网络](https://sebastianraschka.com/teaching/pytorch-1h/)教程
- 如果您遵循了上一节，您应该已经安装了PyTorch
- 手动检查您的PyTorch安装是否支持GPU；查看您的机器支持什么：

In [5]:
import torch

print(f"PyTorch version {torch.__version__}")

if torch.cuda.is_available():
    print("CUDA GPU")
elif torch.mps.is_available():
    print("Apple Silicon GPU")
else:
    print("Only CPU")

PyTorch version 2.7.1
Apple Silicon GPU


- Depending on the chapter, code will automatically use NVIDIA GPU if available, otherwise run on CPU (or Apple Silicon GPU if recommended for a particular section or chapter)
- Chapters 2-4 can be executed in a reasonable time on a CPU
- Code in chapters 5-7 will be very slow when executed on a CPU, and a GPU with NVIDIA is recommended for these chapters (more on the exact resource needs in those upcoming chapters)
- My personal preference is [Lightning AI Studio](https://lightning.ai/), which offers users free compute credits after the sign-up and verification process; alternatively, [Google Colab](https://colab.research.google.com/) is another good choice
- 根据章节，如果支持的话，代码将自动使用NVIDIA GPU，否则在CPU上运行（或如果特定章节或章节推荐，则使用Apple Silicon GPU）
- 2-4章可以在CPU上合理的时间内执行
- 如果在CPU上执行，第5-7章的代码将非常慢，因此建议使用NVIDIA GPU（在即将到来的章节中，更详细地介绍了确切资源需求）
- 我的首选是[Lightning AI Studio](https://lightning.ai/)，它为用户提供了免费计算积分，在注册和验证过程之后；或者，[Google Colab](https://colab.research.google.com/)是另一个不错的选择

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F04_raschka.webp" width="500px">

- See [../02_setup-tips/gpu-instructions.md](../02_setup-tips/gpu-instructions.md) for cloud compute recommendations if needed
- But for now, there is no need to use GPUs yet; the first chapters run fine on non-GPU hardware
- 如果需要，请参阅[../02_setup-tips/gpu-instructions.md](../02_setup-tips/gpu-instructions.md)以获取云计算推荐
- 但现在还没有必要使用GPU；前几章在非GPU硬件上运行良好

&nbsp;
## 2.4 Preparing input texts for LLMs 准备LLMs的输入文本

- In this section, we learn how to use a tokenizer; we use it to convert (encode) input text into a token ID representation as input to the LLM
- We also use the tokenizer to convert (decode) the LLM output back into a human-readable text representation
- 在本节中，我们学习如何使用分词器；我们使用它将（编码）输入文本转换为LLM的标记ID表示作为输入
- 我们还使用分词器将（解码）LLM输出转换回人类可读的文本表示

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F05_raschka.webp?1" width="500px">

- As mentioned earlier, implementing the LLM and tokenizer from scratch is outside the scope of this book, which is focused on implementing reasoning methods from scratch on top of an existing LLM and tokenizer
- In this book, we will work with a pre-trained LLM that we will load in the next section; here, we load the tokenizer that goes with it
- I prepared a `reasoning_from_scratch` Python package that provides the base LLM and the corresponding tokenizer, which I coded with the help of the [`tokenizers`](https://github.com/huggingface/tokenizers) Python library package
- The `reasoning_from_scratch` package code is part of this book's supplementary code, and it should already be installed based on the instructions in section 2.2
- 如前所述，从头开始实现LLM和分词器超出了本书的范围，本书专注于在现有LLM和分词器之上从头开始实现推理方法
- 在本书中，我们将使用一个预训练的LLM，我们将在下一节中加载它；在这里，我们加载与之对应的分词器
- 我准备了一个`reasoning_from_scratch`Python包，它提供了基础LLM和相应的分词器，这里使用[`tokenizers`](https://github.com/huggingface/tokenizers)Python库包
- `reasoning_from_scratch`包代码是本书补充代码的一部分，根据第2.2节中的说明，它应该已经安装

- Next, we download the tokenizer files (this is a tokenizer for the Qwen3 base LLM, but more on that in the next section):
- 接下来，我们下载分词器文件（这是一个用于Qwen3基础LLM的分词器，更多内容将在下一节中介绍）：

In [6]:
from reasoning_from_scratch.qwen3 import download_qwen3_small

download_qwen3_small(kind="base", tokenizer_only=True, out_dir="qwen3")

✓ qwen3/tokenizer-base.json already up-to-date


- Now, we can load the tokenizer settings from the tokenizer file into the `Qwen3Tokenizer`:
- 现在，我们可以将分词器文件中的分词器设置加载到`Qwen3Tokenizer`中：

In [7]:
from pathlib import Path
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer

tokenizer_file_path = Path("qwen3") / "tokenizer-base.json"
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_file_path)

- Since we haven't loaded the LLM itself yet, we will do a simpler round-trip: we encode the text into token IDs and then encode it back into its string representation:
- 由于我们还没有加载LLM本身，我们将进行一个简单的往返：我们将文本编码为标记ID，然后将其编码回其字符串表示：

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F06_raschka.webp" width="500px">

In [8]:
prompt = "Explain large language models."
input_token_ids_list = tokenizer.encode(prompt)

In [9]:
for i in input_token_ids_list:
    print(f"{i} --> {tokenizer.decode([i])}")

840 --> Ex
20772 --> plain
3460 -->  large
4128 -->  language
4119 -->  models
13 --> .


In [10]:
text = tokenizer.decode(input_token_ids_list)
print(text)

Explain large language models.


- In case of the `Qwen3Tokenizer`, there are about 151 thousand unique tokens (vocabulary size)
- `Qwen3Tokenizer`的情况下，大约有15.1万个唯一标记（词汇量）

- Additional resources on tokenization:
- 其他关于分词的资源：
  - [Build a Large Language Model (from Scratch)](https://mng.bz/M96o) chapter 2
  - [Implementing A Byte Pair Encoding (BPE) Tokenizer From Scratch](https://sebastianraschka.com/blog/2025/bpe-from-scratch.html)

&nbsp;
## 2.5 Loading pre-trained models 加载预训练模型

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F07_raschka.webp" width="500px">

- As hinted at in the previous section, when loading the tokenizer, this book uses Qwen3 0.6B; after thinking long and hard about which open weight base model to use, I opted for Qwen3 because
  - Qwen3 is the leading open-weight model in terms of modeling performance as of this writing
  - Qwen3 0.6B is more memory efficient than Llama 3 1B
  - There's both a base model (which we focus on for reasoning model development) and an official reasoning variant that we can use as a reference model
- 如上一节中提到的那样，加载分词器时，本书使用Qwen3 0.6B；在深思熟虑后，我选择了Qwen3，因为
  - Qwen3是撰写本文时性能领先的开放权重模型
  - Qwen3 0.6B比Llama 3 1B更节省内存
  - 有一个基础模型（我们专注于推理模型开发）和一个官方推理变体，我们可以将其用作参考模型
- (Note that the canonical spelling does not include a whitespace in "Qwen3" whereas it includes one in "Llama 3")
- (请注意，"Qwen3"的规范拼写中没有空格，而"Llama 3"中有一个空格)
- In the spirit of "from-scratch" we are using a reimplementation of Qwen3 that I wrote in pure PyTorch without any external LLM library dependencies; this from-scratch implementation is compatible with the original Qwen3 model weights
- 在"从头开始"的精神下，我们使用了一个纯PyTorch的重写，我没有使用任何外部LLM库依赖项；这个从头开始的实现与原始Qwen3模型权重兼容
- However, we will not go over the Qwen3 code implementation in this book as this would be a whole book by itself (similar to my [Build A Large Language Model (From Scratch)](https://github.com/rasbt/LLMs-from-scratch) book; instead, this book (Build A Reasoning Model From Scratch) focuses on implementing reasoning methods from scratch on top of a base model (here Qwen3)
- 然而，本书不会介绍Qwen3代码实现，因为这本身就是一本完整的书（类似于我的[从头开始构建大型语言模型](https://github.com/rasbt/LLMs-from-scratch)书；相反，本书（从头开始构建推理模型）专注于在基础模型（这里为Qwen3）之上实现推理方法
- See appendix C for the Qwen3 model code
- See appendix D for loading the reasoning variant and larger Qwen3 models
- See the Qwen3 [GitHub repository](https://github.com/QwenLM/Qwen3) and [technical report](https://arxiv.org/abs/2505.09388) for (even) more details
- 附录C中介绍了Qwen3模型代码
- 附录D中介绍了加载推理变体和更大的Qwen3模型
- 参见Qwen3 [GitHub存储库](https://github.com/QwenLM/Qwen3)和[技术报告](https://arxiv.org/abs/2505.09388)以获取更多详细信息

- The model is purposefully small (but still very capable) to run on consumer hardware
- It runs fine on CPU, NVIDIA GPUs (`\"cuda\"`), Apple Silicon GPUs (`\"mps\"`), and Intel GPUs (`\"xpu\"`); more about the performance trade-offs later in this chapter
- 模型故意很小（但仍然非常强大），可以在消费级硬件上运行
- 它可以在CPU、NVIDIA GPU（`"cuda"`）、Apple Silicon GPU（`"mps"`）和Intel GPU（`"xpu"`）上运行；更多关于性能权衡的内容将在本章后面介绍

In [ ]:
def get_device():
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using NVIDIA CUDA GPU")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple Silicon GPU (MPS)")
    elif torch.xpu.is_available():
        device = torch.device("xpu")
        print("Using Intel GPU (XPU)")
    else:
        device = torch.device("cpu")
        print("Using CPU")
    return device

device = get_device()

Using Apple Silicon GPU (MPS)


- I recommend running the code on `"cpu"` on the first run-through, so we hardcode the device below: 
- 我建议在第一次运行时在`"cpu"`上运行代码，因此我们在下面硬编码了设备：

In [12]:
# Recommended: Use CPU on the first run-through
device = torch.device("cpu")

- Then, we download the file containing the pre-trained model weights, which is approximately 1.5 GB in size:
- 然后，我们下载包含预训练模型权重的文件，该文件大小约为1.5 GB：

In [13]:
download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")

✓ qwen3/qwen3-0.6B-base.pth already up-to-date
✓ qwen3/tokenizer-base.json already up-to-date


- The architectural structure of the Qwen3 0.6B model we are loading is shown below for readers who are familiar with LLM architectures, but note that for this book, it's **not** essential or important to understand this architecture as we are not modifying but rather adding reasoning techniques on top in later chapters
- 我们正在加载的Qwen3 0.6B模型的架构结构如下所示，供熟悉LLM架构的读者参考，但请注意，对于本书来说，了解这个架构并不是必需的，因为我们不会修改，而是在后续章节中在上面添加推理技术

- I coded the Qwen3 model architecture from scratch for the [reasoning-from-scratch](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/qwen3.py) Python package contained in this code repository; the source code is also shown in appendix C; but again, this is only as a bonus for those who are curious, and it's not necessary to look at or understand these internals to follow the rest of the book
- 我从头编写了Qwen3模型的结构在[reasoning-from-scratch](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/qwen3.py)代码库中包含Qwen3的Python包；源代码也显示在附录C中；但再次强调，这只是为了感兴趣的人的额外奖励，并不需要查看或理解这些内部细节来阅读本书的其余部分


In [14]:
from reasoning_from_scratch.qwen3 import Qwen3Model, QWEN_CONFIG_06_B

model_file = Path("qwen3") / "qwen3-0.6B-base.pth"

model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_file))

model.to(device)

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F08_raschka.webp" width="300px">

&nbsp;
## 2.6 Understanding the sequential LLM text generation process 理解顺序LLM文本生成过程

- In this section, we code a simple wrapper function so we can use the LLM to generate text (we will extend this function with extra functionality in chapter 4)
- 在本节中，我们编写一个简单的包装函数，以便我们可以使用LLM生成文本（我们将在第4章中扩展此函数以添加额外功能）

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F09_raschka.webp?1" width="500px">

- LLMs generate one word at a time:
- LLMs一次生成一个单词：

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F10_raschka.webp?2" width="500px">

- The figure above is a simplification, only showing the newly generated word; the figure below zooms in on the first iteration:
- 上图是一个简化图，只显示了新生成的单词；下图放大了第一次迭代：

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F11_raschka.webp" width="3b00px">

注：以下代码unsqueeze和squeeze是torch的tensor操作，用于增加或减少tensor的维度

In [15]:
example = torch.tensor([1, 2, 3]) 
print(example)
print(example.unsqueeze(0))

tensor([1, 2, 3])
tensor([[1, 2, 3]])


In [16]:
example = torch.tensor([[1, 2, 3]]) 
print(example)
print(example.squeeze(0))

tensor([[1, 2, 3]])
tensor([1, 2, 3])


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F11_raschka.webp?2" width="300px">

In [ ]:
prompt = "Explain large language models."
input_token_ids_list = tokenizer.encode(prompt)
print(f"Number of input tokens: {len(input_token_ids_list)}")

input_tensor = torch.tensor(input_token_ids_list)
input_tensor_fmt = input_tensor.unsqueeze(0).to(device)

output_tensor = model(input_tensor_fmt)
output_tensor_fmt = output_tensor.squeeze(0)
print(f"Formatted Output tensor shape: {output_tensor_fmt.shape}")

Number of input tokens: 6
Formatted Output tensor shape: torch.Size([6, 151936])


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F12_raschka.webp" width="500px">

In [18]:
last_token = output_tensor_fmt[-1].detach()
print(last_token)

tensor([ 7.3750,  2.0312,  8.0000,  ..., -2.5469, -2.5469, -2.5469],
       dtype=torch.bfloat16)


In [19]:
print(last_token.argmax(dim=-1, keepdim=True))

tensor([20286])


In [20]:
print(tokenizer.decode([20286]))

 Large


In [21]:
example = torch.tensor([-2, 1, 3, 1])
print(torch.max(example))
print(torch.argmax(example))

tensor(3)
tensor(2)


&nbsp;
## 2.7 Coding a minimal text generation function 编写一个最小的文本生成函数


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F13_raschka.webp" width="500px">

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F14_raschka.webp?2" width="500px">

- The `generate_text_basic` function implements this sequential text generation process:
- `generate_text_basic`函数实现了这个顺序文本生成过程：

In [22]:
@torch.inference_mode()
def generate_text_basic(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None
):
    input_length = token_ids.shape[1]
    model.eval()

    for _ in range(max_new_tokens):
        out = model(token_ids)[:, -1]
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        # Stop if all sequences in the batch have generated EOS
        if (eos_token_id is not None
                and torch.all(next_token == eos_token_id)):
            break

        token_ids = torch.cat([token_ids, next_token], dim=1)
    return token_ids[:, input_length:]

- Let's use it to generate a 100-token response to a simple "Explain large language models in 2 sentences." prompt to see how it works (we get to the reasoning parts in later chapters)
- The following code will be slow and can take 1-3 minutes to complete, depending on your computer (we will speed it up in later sections) 
- 让我们使用它来生成一个100个令牌的响应，以回答一个简单的“用两句话解释大型语言模型”提示，以了解它是如何工作的（我们将在后面的章节中讨论推理部分）
- 以下代码将很慢，并且可能需要1-3分钟才能完成，具体取决于您的计算机（我们将在后面的部分中加快速度）

In [23]:
prompt = "Explain large language models in a single sentence."
input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device=device
    ).unsqueeze(0)

max_new_tokens = 100
output_token_ids_tensor = generate_text_basic(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
)
output_text = tokenizer.decode(
    output_token_ids_tensor.squeeze(0).tolist()
)
print(output_text)

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.<|endoftext|>Human language is a complex and dynamic system that has evolved over millions of years to enable effective communication and social interaction. It is composed of a vast array of symbols, including letters, numbers, and words, which are used to convey meaning and express thoughts and ideas. The evolution of language has


- Notice that the LLM follows the instruction quite well, but the response becomes nonsensical/off-topic after `<|endoftext|>`, which is a token used as a delimiter between different documents during training
- When using the LLM, we want it to stop generating after encountering this token
- 注意到LLM很好地遵循了指令，但在`<|endoftext|>`之后，响应变得毫无意义/脱离主题，这是在训练期间用于分隔不同文档的token
- 当使用LLM时，我们希望它在遇到这个token后停止生成

In [24]:
print(tokenizer.encode("<|endoftext|>"))

[151643]


- For convenience, this token ID is stored as a tokenizer attribute (eos = end of sequence):
- 为了方便，这个token ID被存储为tokenizer属性（eos = end of sequence）：

In [25]:
print(tokenizer.eos_token_id)

151643


- We can use it to tell the LLM (or rather the `generate_text_basic` function) when to stop generating text
- 我们可以使用它来告诉LLM（或者更确切地说，`generate_text_basic`函数）何时停止生成文本

In [26]:
output_token_ids_tensor = generate_text_basic(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
)

output_text = tokenizer.decode(
    output_token_ids_tensor.squeeze(0).tolist()
)
print(output_text)

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.


- The response above is what you get when running to code on CPU, the generated text may differ slightly differ depending on the device
- 上面的响应是在CPU上运行代码时得到的，生成的文本可能会因设备而略有不同

- Before we wrap up this section and see how we can speed up the code, let's implement a simple benchmarking function to track the computational performance
- 在我们结束本节并了解如何加快代码之前，让我们实现一个简单的基准测试函数来跟踪计算性能

In [27]:
def generate_stats(output_token_ids, tokenizer, start_time, end_time):
    total_time = end_time - start_time
    print(f"Time: {total_time:.2f} sec")
    print(f"{int(output_token_ids.numel() / total_time)} tokens/sec")

    if torch.cuda.is_available():
        max_mem_bytes = torch.cuda.max_memory_allocated()
        max_mem_gb = max_mem_bytes / (1024 ** 3)
        print(f"Max memory allocated: {max_mem_gb:.2f} GB")

    output_text = tokenizer.decode(output_token_ids.squeeze(0).tolist())
    print(f"\n{output_text}")

In [28]:
import time

start_time = time.time()
output_token_ids_tensor = generate_text_basic(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
)
end_time = time.time()


generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

Time: 9.23 sec
4 tokens/sec

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.


&nbsp;
## 2.8 Faster inference via KV caching 通过KV缓存加快推理速度

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F15_raschka.webp?2" width="500px">

- Note that the code in this book emphasizes code readability, and a whole separate book can be written about optimizations
- Here, we look at an engineering trick called "KV caching" (KV refers to the keys and values inside the attention mechanism of the LLM)
- If you are unfamiliar with these terms, don't worry, all you need to know is that there is a way we can store (cache) intermediate values that are reused in each iteration
- 请注意，本书中的代码强调代码的可读性，可以专门写一本书来介绍优化
- 在这里，我们看一下一种称为“KV缓存”的工程技巧（KV指的是LLM注意力机制中的键和值）
- 如果您对这些术语不熟悉，不用担心，您需要知道的是，有一种方法可以存储（缓存）在每次迭代中重复使用的中间值

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F16_raschka.webp" width="500px">

- For more details on the mechanics of KV caching, see my [Understanding and Coding the KV Cache in LLMs from Scratch](https://magazine.sebastianraschka.com/p/coding-the-kv-cache-in-llms) article
- Below is a modified version of the `generate_text_basic` function that uses a KV cache
- 有关KV缓存的更多细节，请参阅我的<从头开始理解和编码LLM中的KV缓存>[Understanding and Coding the KV Cache in LLMs from Scratch](https://magazine.sebastianraschka.com/p/coding-the-kv-cache-in-llms)文章
- 以下是`generate_text_basic`函数的一个修改版本，它使用了KV缓存

In [29]:
from reasoning_from_scratch.qwen3 import KVCache

@torch.inference_mode()
def generate_text_basic_cache(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None
):

    input_length = token_ids.shape[1] 
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()

    out = model(token_ids, cache=cache)[:, -1]
    for _ in range(max_new_tokens):
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        if (eos_token_id is not None 
               and torch.all(next_token == eos_token_id)):
            break

        token_ids = torch.cat([token_ids, next_token], dim=1)
        out = model(next_token, cache=cache)[:, -1]

    return token_ids[:, input_length:]

- The usage is similar to before:
- 使用方法与之前类似：

In [30]:
start_time = time.time()

output_token_ids_tensor = generate_text_basic_cache(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id,
)
end_time = time.time()

generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

Time: 1.40 sec
29 tokens/sec

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.


- As we can see, it is magnitudes faster than before (28 tokens/sec instead of 4 tokens/sec; run on a Mac Mini M4 CPU)
- 如我们所见，它比之前快得多（28 tokens/sec而不是4 tokens/sec；在Mac Mini M4 CPU上运行）

&nbsp;
## 2.9 Faster inference via PyTorch model compilation 通过PyTorch模型编译加快推理速度

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F17_raschka.webp?2" width="500px">

- Another technique to speed up the model inference (text generation) by a lot is using `torch.compile`
- Note that this currently doesn't work on MPS (Apple Silicon GPU) devices due to `InductorError`
- The usage is simple, we just call `torch.compile` on the model (see [the documentation](https://docs.pytorch.org/docs/stable/torch.compiler.html) for additional options)
- 另一种通过大量使用`torch.compile`来加快模型推理（文本生成）的技术
- 请注意，由于`InductorError`，这目前不适用于MPS（Apple Silicon GPU）设备
- 使用方法很简单，我们只需在模型上调用`torch.compile`（有关其他选项，请参阅文档[the documentation](https://docs.pytorch.org/docs/stable/torch.compiler.html)）

In [31]:
if device.type == "mps":
    print(f"`torch.compile` is not supported for the {model.__class__.__name__} model on MPS (Apple Silicon) as of this writing.")
    model_compiled = model
else:
    model_compiled = torch.compile(model)
    # Assignment so that notebook doesn't stop here if someone uses "Run All Cells"

- The first iteration can be a bit slow as it does the initial compilation and optimization; hence, we repeat the text generation multiple times
- First, let's start with the non-cached version (this can be a bit slow and might take xx minutes)
- 第一次迭代可能会慢一些，因为它需要初始编译和优化；因此，我们多次重复文本生成
- 首先，让我们从非缓存版本开始（这可能有点慢，可能需要xx分钟）

In [32]:
for i in range(3):
    start_time = time.time()
    output_token_ids_tensor = generate_text_basic(
        model=model_compiled,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    )
    end_time = time.time()

    if i == 0:
        print("Warm-up run")
    else:
        print(f"Timed run {i}:")
    generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

    print(f"\n{30*'-'}\n")

Warm-up run
Time: 27.70 sec
1 tokens/sec

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

------------------------------

Timed run 1:
Time: 7.09 sec
5 tokens/sec

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

------------------------------

Timed run 2:
Time: 7.19 sec
5 tokens/sec

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

------------------------------



- As we can see above, with 5 tokens/sec, this is only marginally faster than before (4 tokens/sec)
- Let's now see how well the KV cache version does
- 如上所示，以5 tokens/sec的速度，这比之前稍微快一点（4 tokens/sec）
- 现在让我们看看KV缓存版本的效果如何

In [33]:
for i in range(3):
    start_time = time.time()
    output_token_ids_tensor = generate_text_basic_cache(
        model=model_compiled,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    )
    end_time = time.time()

    if i == 0:
        print("Warm-up run")
    else:
        print(f"Timed run {i}:")
    generate_stats(
        output_token_ids_tensor, tokenizer, start_time, end_time
    )

    print(f"\n{30*'-'}\n")

Warm-up run
Time: 29.87 sec
1 tokens/sec

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

------------------------------

Timed run 1:
Time: 0.60 sec
68 tokens/sec

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

------------------------------

Timed run 2:
Time: 0.62 sec
66 tokens/sec

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

------------------------------



- As we can see, the compilation resulted in a substantial 2x speed-up (64 tokens/sec versus 30 tokens/sec)
- Below is a table with additional results
- 如我们所见，编译导致了显著的2倍速度提升（64 tokens/sec与30 tokens/sec）
- 以下是其他结果表

| Model      | Mode              | Hardware        | Tokens/sec    | GPU Memory (VRAM) |
|------------|-------------------|-----------------|---------------|-------------------|
| Qwen3Model | Regular           | Mac Mini M4 CPU | 6             | -                 |
| Qwen3Model | Regular compiled  | Mac Mini M4 CPU | 6             | -                 |
| Qwen3Model | KV cache          | Mac Mini M4 CPU | 28            | -                 |
| Qwen3Model | KV cache compiled | Mac Mini M4 CPU | 68            | -                 |
|            |                   |                 |               |                   |
| Qwen3Model | Regular           | Mac Mini M4 GPU | 17            | -                 |
| Qwen3Model | Regular compiled  | Mac Mini M4 GPU | InductorError | -                 |
| Qwen3Model | KV cache          | Mac Mini M4 GPU | 18            | -                 |
| Qwen3Model | KV cache compiled | Mac Mini M4 GPU | InductorError | -                 |
|            |                   |                 |               |                   |
| Qwen3Model | Regular           | NVIDIA H100 GPU | 51            | 1.55 GB           |
| Qwen3Model | Regular compiled  | NVIDIA H100 GPU | 164           | 1.81 GB           |
| Qwen3Model | KV cache          | NVIDIA H100 GPU | 48            | 1.52 GB           |
| Qwen3Model | KV cache compiled | NVIDIA H100 GPU | 141           | 1.81 GB           |

- Note that we ran all the examples with a single prompt (i.e., a batch size of 1); if you are curious about batched inference, see appendix E
- 请注意，我们使用单个提示（即，批处理大小为1）运行了所有示例；如果您对批处理推理感兴趣，请参阅附录E

&nbsp;
## Summary 总结

- No code in this section
- 本节没有代码